## Without web search

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [5]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
model = init_chat_model(
    model="arc:lite",
    model_provider="openai",
    base_url=os.getenv("RADAR_OPEN_MODEL_BASE_URL"),
    api_key=os.getenv("RADAR_OPEN_MODEL_API_KEY"),
)
agent = create_agent(
    model=model
)

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [8]:
print(response['messages'][-1].content)

My knowledge cutoff is **January 2025**. Therefore, I do not have information about events or developments that have occurred after that date unless they are provided to me in the context of our conversation.


## Add web search tool

In [14]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current President of India?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'http://www.presidentofindia.gov.in/Profile',
   'title': 'Profile | President of India',
   'content': '#### Smt. Droupadi MurmuThe President of India. #### Profile of the President. Smt. Droupadi Murmu was sworn in as the 15th President of India on July 25, 2022. Prior to this, she served as the 9th Governor of the state of Jharkhand from 2015 to 2021. Smt. Droupadi Murmu was born on June 20, 1958, to a Santhal tribal family in Uparbeda in the remote Mayurbhanj district of Odisha. She became the first girl in her village to pass the matriculation examination and to earn a degree, graduating with a Bachelor of Arts degree in Political Science and Economics in 1979 from Ramadevi Women’s College, Bhubaneswar. From 1979 to 1983, Smt. Murmu served as a Junior Assistant in the Irrigation and Power Department, Government of Odisha. Smt. Droupadi Murmu, a lea

In [19]:
agent = create_agent(
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [20]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='19ca9b08-d262-44b8-abf1-aae16507a53a'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 69, 'total_tokens': 88, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'arc:lite', 'system_fingerprint': 'vllm-0.23.0-406383e7', 'id': 'chatcmpl-8f2dc822ca90fb6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f5c2d-4e91-7b61-8fac-ae96d38a307d-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco'}, 'id': 'chatcmpl-tool-980c522c39880fca', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 19, 'total_tokens': 88, 'input_token_details': {}, 'output_token_details': {}}),
 ToolMessage(content='{"query": "current mayor of San Francisco", "fol

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r